# Diagnostic Multi-Source avec Deep Sets (Mean + Max Pooling) sur Événements Bruts

Ce notebook implémente une architecture **Deep Sets** (`Mean + Max Pooling`) en PyTorch sur le dataset événementiel (`detection_error_src.csv`) :
- Chaque ligne d'événement (0 à $n$ opérations machines ou manuelles par câble) est encodée individuellement.
- Les représentations des événements d'un **groupe quelconque** (par défaut `Batch_ID`, mais adaptable à la volée à d'autres regroupements comme `Cable_ID`, `Operation_Type`, ou un sous-ensemble d'événements **sans réentraîner**) sont combinées via **Mean Pooling + Max Pooling**.
- Une tête multi-label produit **21 probabilités** (`MACH_01..05`, `OP_01..08`, `SENS_01..04`, `INSP_01..04`) comparées ensuite au seuil de décision $C$.

In [13]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, hamming_loss

# Reproductibilité et appareil (CPU ou GPU)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Appareil utilisé :', device)

Appareil utilisé : cpu


## 1. Chargement et préparation des variables événementielles (sans fuite de cible)

In [14]:
DATA_PATH = 'D:/S5/Projet_d_expertise/projet/data/processed/detection_error_src.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes')

MACHINES = ['MACH_01', 'MACH_02', 'MACH_03', 'MACH_04', 'MACH_05']
OPERATORS = [f'OP_{i:02d}' for i in range(1, 9)]
SENSORS = ['SENS_01', 'SENS_02', 'SENS_03', 'SENS_04']
INSPECTORS = ['INSP_01', 'INSP_02', 'INSP_03', 'INSP_04']
ALL_ENTITIES = MACHINES + OPERATORS + SENSORS + INSPECTORS
TARGET_COLS = [f'Target_Prob_{e}' for e in ALL_ENTITIES]

# Variables numériques observables au niveau de l'événement
NUM_COLS = [
    'Step_Order', 'Total_Cable_Steps', 'Num_Machine_Ops', 'Num_Manual_Ops',
    'Hours_Into_Shift',
    'Machine_Hours_Since_Maint', 'Machine_Vibration_mm_s', 'Machine_Temp_C', 'Machine_Cycle_Time_sec',
    'Operator_Exp_Months', 'Operator_Handling_Time_sec',
    'Sensor_Calibration_Drift', 'Sensor_Anomaly_Score', 'Sensor_Error_Flag',
    'Inspection_Duration_sec', 'Human_Error_Flag',
    'Step_Observed_Error', 'Confirmed_Physical_Defect'
]

# Variables catégorielles observables encodées en one-hot
CAT_COLS = ['Shift', 'Operation_Mode', 'Operation_Type', 'Machine_ID', 'Operator_ID', 'Sensor_ID', 'Inspector_ID', 'Control_Mode']

df_num = df[NUM_COLS].fillna(0.0).copy()
df_cat = pd.get_dummies(df[CAT_COLS].fillna('NONE'), dtype=np.float32)

X_event_raw = pd.concat([df_num, df_cat], axis=1)
X_event_raw['Batch_ID'] = df['Batch_ID']
X_event_raw['Cable_ID'] = df['Cable_ID']
X_event_raw['Operation_Type'] = df['Operation_Type']

FEATURE_COLS = [c for c in X_event_raw.columns if c not in ['Batch_ID', 'Cable_ID', 'Operation_Type']]
print(f'Nombre de features par événement : {len(FEATURE_COLS)}')

Dataset chargé : 58970 lignes, 52 colonnes
Nombre de features par événement : 60


## 2. Découpage par lot (`Batch_ID`), normalisation et Dataset PyTorch

In [15]:
batch_targets = df.groupby('Batch_ID', sort=True)[TARGET_COLS].first()
all_batches = batch_targets.index.tolist()

train_batches, test_batches = train_test_split(all_batches, test_size=0.25, random_state=SEED)

# Standardisation ajustée uniquement sur les événements du train
train_mask = X_event_raw['Batch_ID'].isin(train_batches)
scaler = StandardScaler()
X_scaled = X_event_raw[FEATURE_COLS].copy()
X_scaled.loc[train_mask] = scaler.fit_transform(X_event_raw.loc[train_mask, FEATURE_COLS])
X_scaled.loc[~train_mask] = scaler.transform(X_event_raw.loc[~train_mask, FEATURE_COLS])
X_scaled['Batch_ID'] = X_event_raw['Batch_ID']

def build_group_samples(df_feat: pd.DataFrame, group_col: str, targets_df: pd.DataFrame = None):
    """Construit une liste d'ensembles d'événements selon n'importe quelle colonne de regroupement."""
    samples = []
    grouped = {k: v[FEATURE_COLS].values.astype(np.float32) for k, v in df_feat.groupby(group_col, sort=False)}
    for gid, arr in grouped.items():
        if len(arr) == 0:
            continue
        y_vec = targets_df.loc[gid].values.astype(np.float32) if (targets_df is not None and gid in targets_df.index) else None
        samples.append((gid, arr, y_vec))
    return samples

train_samples = build_group_samples(X_scaled[X_scaled['Batch_ID'].isin(train_batches)], 'Batch_ID', batch_targets)
test_samples = build_group_samples(X_scaled[X_scaled['Batch_ID'].isin(test_batches)], 'Batch_ID', batch_targets)

class EventSetDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        gid, feats, target = self.samples[idx]
        return gid, torch.tensor(feats, dtype=torch.float32), (torch.tensor(target, dtype=torch.float32) if target is not None else None)

def collate_variable_sets(batch):
    """Concatène les événements du batch de groupes et crée un index d'appartenance au groupe."""
    gids, feat_list, target_list = zip(*batch)
    group_ids = []
    for i, f in enumerate(feat_list):
        group_ids.append(torch.full((f.size(0),), i, dtype=torch.long))
    x_cat = torch.cat(feat_list, dim=0)
    g_idx = torch.cat(group_ids, dim=0)
    y_out = torch.stack(target_list, dim=0) if target_list[0] is not None else None
    return gids, x_cat, g_idx, len(feat_list), y_out

train_loader = DataLoader(EventSetDataset(train_samples), batch_size=16, shuffle=True, collate_fn=collate_variable_sets)
test_loader = DataLoader(EventSetDataset(test_samples), batch_size=32, shuffle=False, collate_fn=collate_variable_sets)

C:\Users\hp\AppData\Local\Temp\ipykernel_5580\2724789436.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.18197196 -0.54621595  0.08954007 ...  1.36105209  1.9968081
  2.63256411]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_scaled.loc[train_mask] = scaler.fit_transform(X_event_raw.loc[train_mask, FEATURE_COLS])
C:\Users\hp\AppData\Local\Temp\ipykernel_5580\2724789436.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.49489907 -0.49489907 -0.49489907 ...  1.57200487  1.57200487
  1.57200487]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_scaled.loc[train_mask] = scaler.fit_transform(X_event_raw.loc[train_mask, FEATURE_COLS])
C:\Users\hp\AppData\Local\Temp\ipykernel_5580\2724789436.py:10: FutureWarning: Setting an item 

## 3. Architecture Deep Sets (Mean + Max Pooling)

In [16]:
class AttentionDeepSetsDiagnostic(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 128, num_targets: int = 21):
        super().__init__()
        # Encodeur par événement avec LayerNorm et GELU pour une meilleure stabilité
        self.phi = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        # Score d'attention par événement et par cible (num_targets têtes d'attention)
        self.attn_score = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, num_targets),
        )

        # Tête de prédiction multi-label combinant contexte moyen + contexte attentif
        self.rho = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim * num_targets, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim * 2, num_targets),
        )
        self.num_targets = num_targets

    def forward(self, x_events: torch.Tensor, group_idx: torch.Tensor, num_groups: int) -> torch.Tensor:
        h = self.phi(x_events)  # (N_events, hidden_dim)
        dev = x_events.device
        dim = h.size(1)

        # 1. Mean Pooling global du groupe
        sum_pool = torch.zeros((num_groups, dim), device=dev, dtype=h.dtype)
        sum_pool.index_add_(0, group_idx, h)
        counts = torch.zeros((num_groups, 1), device=dev, dtype=h.dtype)
        counts.index_add_(0, group_idx, torch.ones((h.size(0), 1), device=dev, dtype=h.dtype))
        mean_pool = sum_pool / counts.clamp(min=1.0)  # (num_groups, hidden_dim)

        # 2. Softmax par groupe sur les scores d'attention (N_events, num_targets)
        scores = self.attn_score(h)
        max_scores = torch.full((num_groups, self.num_targets), -1e9, device=dev, dtype=h.dtype)
        max_scores.scatter_reduce_(
            0, group_idx.unsqueeze(1).expand(-1, self.num_targets), scores, reduce="amax", include_self=True
        )
        exp_scores = torch.exp(scores - max_scores[group_idx])
        denom = torch.zeros((num_groups, self.num_targets), device=dev, dtype=h.dtype)
        denom.index_add_(0, group_idx, exp_scores)
        attn_weights = exp_scores / denom[group_idx].clamp(min=1e-8)  # (N_events, num_targets)

        # Agrégation pondérée par cible : (num_groups, num_targets, hidden_dim)
        weighted_h = attn_weights.unsqueeze(-1) * h.unsqueeze(1)
        attn_pool = torch.zeros((num_groups, self.num_targets, dim), device=dev, dtype=h.dtype)
        attn_pool.index_add_(0, group_idx, weighted_h)
        attn_flat = attn_pool.reshape(num_groups, -1)

        # 3. Fusion et prédiction des 21 logits
        combined = torch.cat([mean_pool, attn_flat], dim=1)
        return self.rho(combined)


EPOCHS = 40
model = AttentionDeepSetsDiagnostic(
    in_features=len(FEATURE_COLS), hidden_dim=128, num_targets=len(TARGET_COLS)
).to(device)

# Pondération douce des classes positives rares pour améliorer le F1-Macro
y_train_mat = np.stack([s[2] for s in train_samples])
pos_counts = y_train_mat.sum(axis=0)
neg_counts = len(y_train_mat) - pos_counts
pos_weight = torch.tensor(
    np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 4.0),
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

## 4. Entraînement avec mesure du temps d'exécution

In [17]:
start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for _, x_cat, g_idx, n_grp, y_batch in train_loader:
        x_cat, g_idx, y_batch = x_cat.to(device), g_idx.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(x_cat, g_idx, n_grp)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item() * n_grp

    scheduler.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{EPOCHS:02d} - Loss entraînement : {total_loss / len(train_samples):.4f}")

elapsed = time.perf_counter() - start_time
print(f"Temps total d'entraînement ({EPOCHS} epochs) : {elapsed:.2f} secondes")

Epoch 01/40 - Loss entraînement : 0.6235
Epoch 05/40 - Loss entraînement : 0.4346
Epoch 10/40 - Loss entraînement : 0.3203
Epoch 15/40 - Loss entraînement : 0.2487
Epoch 20/40 - Loss entraînement : 0.1857
Epoch 25/40 - Loss entraînement : 0.1499
Epoch 30/40 - Loss entraînement : 0.1246
Epoch 35/40 - Loss entraînement : 0.1132
Epoch 40/40 - Loss entraînement : 0.1077
Temps total d'entraînement (40 epochs) : 70.50 secondes


## 5. sauvegarder le modèle Deep Sets

In [19]:
import torch
CHECKPOINT_PATH = "../models/detection_error_src.pt"
THRESHOLD = 0.30
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "in_features": len(FEATURE_COLS),
        "hidden_dim": 128,
        "num_targets": len(TARGET_COLS),
        "scaler": scaler,
        "feature_cols": FEATURE_COLS,
        "entities": ALL_ENTITIES,
        "threshold": THRESHOLD,
    },
    CHECKPOINT_PATH,
)

print(f"Modèle sauvegardé dans : {CHECKPOINT_PATH}")

Modèle sauvegardé dans : ../models/detection_error_src.pt
